# 01 - Eksplorasi Data UMKM dengan Pandas
Notebook ini melakukan data understanding, profiling, quality gate, KPI, agregasi, dan tiga visualisasi.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'run_pipeline.py').exists():
    ROOT = Path('/content')
sys.path.insert(0, str(ROOT))
from run_pipeline import transform_umkm, quality_checks, build_aggregates
CSV_PATH = ROOT / 'data' / 'synthetic_umkm_data.csv'
if not CSV_PATH.exists():
    CSV_PATH = Path('/content/synthetic_umkm_data.csv')
raw = pd.read_csv(CSV_PATH)
df = transform_umkm(raw)
print('Shape:', df.shape)
display(df.head())

In [ ]:
# Profiling schema, missing value, duplikasi, dan statistik
df.info()
display(df.isna().sum().to_frame('missing'))
print('ID duplikat:', df['umkm_id'].duplicated().sum())
display(df.describe(include='all').T)

In [ ]:
# Quality gates
checks = quality_checks(df)
display(pd.DataFrame({'check': checks.keys(), 'passed': checks.values()}))
assert all(checks.values()), 'Ada quality gate yang gagal.'

In [ ]:
# KPI ringkas
kpi = {
    'umkm_count': len(df),
    'avg_monthly_revenue': df['monthly_revenue'].mean(),
    'avg_profit_margin_pct': df['net_profit_margin_pct'].mean(),
    'loss_making_share_pct': (df['net_profit_margin_pct'] < 0).mean() * 100,
    'high_risk_share_pct': (df['operational_risk'] == 'High').mean() * 100,
    'avg_repeat_order_pct': df['repeat_order_rate_pct'].mean(),
}
pd.Series(kpi, name='value')

In [ ]:
# Tiga agregasi bisnis utama
aggregates = build_aggregates(df)
display(aggregates['class_summary'])
display(aggregates['tenure_summary'])
display(aggregates['digital_adoption_summary'])
display(aggregates['latency_summary'])

In [ ]:
# Visualisasi 1: distribusi kelas
class_summary = aggregates['class_summary']
class_summary.plot.bar(x='business_class', y='umkm_count', legend=False, figsize=(9, 4), color=['#C44E52','#DD8452','#4C72B0','#55A868'])
plt.title('Distribusi Kelas Kinerja UMKM'); plt.ylabel('Jumlah UMKM'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Visualisasi 2: kinerja finansial per kelas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_summary.plot.bar(x='business_class', y='avg_monthly_revenue', legend=False, ax=axes[0], color='#4C72B0')
class_summary.plot.bar(x='business_class', y='avg_profit_margin_pct', legend=False, ax=axes[1], color='#55A868')
axes[0].set_title('Rata-rata Pendapatan Bulanan'); axes[0].set_ylabel('Rupiah')
axes[1].set_title('Rata-rata Margin Laba'); axes[1].set_ylabel('Persen'); axes[1].axhline(0, color='black', lw=.8)
for ax in axes: ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
# Visualisasi 3: adopsi digital dan repeat order
adoption = aggregates['digital_adoption_summary']
adoption.plot.bar(x='digital_adoption_band', y='avg_repeat_order_pct', legend=False, figsize=(9, 4), color='#8172B2')
plt.title('Repeat Order menurut Tingkat Adopsi Digital'); plt.ylabel('Rata-rata Repeat Order (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Catatan interpretasi
Dataset tidak memiliki tanggal observasi. `tenure_band` dipakai sebagai cohort temporal dan tidak boleh dibaca sebagai tren kalender. Korelasi/selisih antarkelompok adalah asosiasi, bukan bukti kausal.